In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Gold").getOrCreate()

In [0]:
gold_df = spark.table('workspace.default.silver_anime')
display(gold_df)

**Engagement Score**<br>
How much the audience is **actively interested** in the anime.
```
High members = many people watched it
High favorites = people loved it enough to save it
```


In [0]:
from pyspark.sql.functions import col, log

gold_df = gold_df.withColumn(
  "engagement_score",
  log(col("members")) + log(col("favorites")+1)
)

**Score Tier (Label)**<br>
Instead of raw numbers, give each anime a **human readable quality label**.


In [0]:
from pyspark.sql.functions import when

gold_df = gold_df.withColumn("score_tier",
    when(col("score") >= 8.0, "Excellent")
    .when(col("score") >= 7.0, "Good")
    .when(col("score") >= 6.0, "Average")
    .when(col("score") >  0,   "Below Average")
    .otherwise("Unscored")
)

**Popularity Momentum** <br>
How many people **watched but didn't bother rating** it.
```
High ratio = viral/casual watch (like trending shows)
Low ratio  = dedicated fanbase who all rated it
```

In [0]:
from pyspark.sql.functions import when
gold_df = gold_df.withColumn(
    "popularity_momentum",
    when(col("score_tier") == "Unscored", 0)
    .otherwise(col("members") / (col("scored_by") + 1))
)

Score Z-Score (Normalized Score)
<br>
Is this anime's score **above or below average** compared to all anime?
```
score_z =  2.0  → much better than average
score_z =  0.0  → exactly average
score_z = -2.0  → much worse than average
```

In [0]:
from pyspark.sql.functions import avg, stddev

score_stats = gold_df.filter(col("score")>0).select(
  avg("score").alias("mean"),
  stddev("score").alias("std")
).collect()[0]

mean_score = score_stats["mean"]
std_score = score_stats['std']

print(f"Mean Score: {mean_score:.2f}")
print(f"Std Score:  {std_score:.2f}")

gold_df = gold_df.withColumn(
    "score_z",
    when(col("score") > 0, (col("score") - mean_score) / std_score)
    .otherwise(None)   # ← unscored gets null, not -7.82
)


**Retention Proxy**<br>
Out of everyone who watched, how many **loved it enough to favorite it?**
```
High = strong loyal fanbase (Attack on Titan, FMA)
Low  = people watched but didn't care much
```

In [0]:
gold_df = gold_df.withColumn(
  "retention_proxy",
  col("favorites") / (col("members")+1)
)

**Airing Era (Time Period Label)**<br>
Group anime into **time periods** for trend analysis.

In [0]:
gold_df = gold_df.withColumn("airing_era",
    when(col("year") <= 2012, "Digital Boom Era")       # 2010–2012: post-2006 decline recovery
    .when(col("year") <= 2016, "Streaming Revolution")  # 2013–2016: Crunchyroll/Netflix rise, AoT era
    .when(col("year") <= 2020, "Global Expansion Era")  # 2017–2020: anime goes truly global, Demon Slayer
    .otherwise("Modern Renaissance")                    # 2021–2026: post-COVID boom, record productions
)

**Verify the metrics**

In [0]:
display(gold_df.select(
    "title", "score", "members", "favorites",
    "engagement_score", "popularity_momentum",
    "score_z", "retention_proxy",
    "score_tier", "airing_era"
).limit(20))


In [0]:
gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_anime")

print("Gold table saved!")
